# Experiments 10-13

# Environment Preparation

## Package Importing

In [ ]:
# Experiments 10-13 (conventional ML). Per-modality Random Forest classification and majority-vote late fusion: Methods 2.3.4 and Appendix C.2; feature extraction: Appendix B.2.
import sys, os

sys.path.append(os.path.abspath("../env/configs"))  # Add the parent 'env' directory to the system path
from configs import *
from emotion_classification_tasks import EmotionClassificationTask
from psychai.visualization.chart_helper import ChartHelper
from psychai.feature.feature_engineering.feature_engineering_helper import FeatureEngineeringHelper
from psychai.machine_learning.classification.classifier import Classifier
classifier = Classifier()
chart_helper = ChartHelper()
feature_engineering_helper = FeatureEngineeringHelper()
emotion_classification_task = EmotionClassificationTask(classifier=classifier, chart_helper = chart_helper, feature_enginnering_helper = feature_engineering_helper)
list_users = list(range(1, 10)) + list(range(11, 20))

## Data Loading

In [ ]:
if os.path.exists(path_pickle_df_list) and os.path.exists(path_pickle_df_names):
    try:
        all_df = data_loader.load_from_cache(path_pickle_df_list)
        all_df_name = data_loader.load_from_cache(path_pickle_df_names)
        print("Pickle files loaded successfully.")
    except Exception as e:
        print(f"Failed to load pickle files: {e}. Re-running processing.")
        all_df = None
        all_df_name = None
else:
    df_pose = data_loader.get_pose_features_without_landmarks_data_frame(record_path= path_pose, prefix = "pose")
    df_acoustics = data_loader.get_acoustics_df(path_acoustic_target, path_acoustic_baseline, prefix = "acoustics")
    df_speech_text = data_loader.get_speech_text_df(path_speech_text, prefix = "speech_text")
    df_facial = data_loader.get_facial_df(file_target_path = path_facial_target, file_baseline_path = path_facial_baseline, outlier_multiplier= outlier_multiplier, prefix = "facial")
    df_eye_movement = data_loader.get_eye_movement_df(path_eye_movement, prefix = "eye_move")
    df_remote_ppg = data_loader.get_r_ppg_df(record_path= path_r_ppg, df_info = df_pose[df_pose.columns[0:2]], prefix = "r_ppg")
    df_thermal = data_loader.get_thermal_df(path_thermal, prefix = "thermal")
    df_mm_llm = data_loader.get_mm_llm_df(path_mm_llm, prefix = "mm_llm")
    all_df = [df_acoustics,df_speech_text,df_facial,df_pose,df_eye_movement,df_remote_ppg,df_thermal,df_mm_llm]
    all_df_name = ["df_acoustics","df_speech_text","df_facial","df_pose","df_eye_movement","df_remote_ppg","df_thermal","df_mm_llm"]
    data_loader.save_to_cache(all_df, path_pickle_df_list)
    data_loader.save_to_cache(all_df_name, path_pickle_df_names)

In [ ]:
with pd.option_context(
    "display.max_colwidth", None,      # show full cell content
    "display.max_rows", None,          # show all rows
    "display.max_columns", None,       # show all columns
    "display.width", None,             # auto-detect display width
    "display.expand_frame_repr", False # prevent column wrapping
):
    display(all_df[0])

In [ ]:
for index in range(len(all_df)):
    df = all_df[index]
    print(f"table {all_df_name[index]}, number of columns: {len(df.columns.tolist())}")

In [ ]:
# Map numeric to names

four_group_names = {
    1: "ELVT",
    2: "SYMP",
    3: "INSP",
    4: "CTRL"
}

counts = all_df[0]['Group'].map(four_group_names).value_counts()

# Reindex by the key order in group_names
ordered_labels = [four_group_names[k] for k in sorted(four_group_names.keys())]
counts = counts.reindex(ordered_labels)

# Plot
counts.plot(kind='bar')

## Emotion Recognition Task Settings

In [ ]:
# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.
n_kept_list = [15]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [False]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]


# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]


# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

# Single Variate Analysis

In [ ]:
df_multimodal_evaluations = pd.DataFrame()
df_multimodal_predictions = pd.DataFrame()

## 1. Speech Acoustics

### Definition

In [ ]:
df = all_df[0].copy()
df_name = "Accoustics"

# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.
number_of_features = 15  # Fixed number of retained features (matches Methods)
n_kept_list = [number_of_features]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [True]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]
# RFE_model_name_list = ["Random Forest"]

# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]
# default_model_name_list = ["Random Forest"]

# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

In [ ]:
features_to_plot= ['median_acoustics-mfcc2_sma3_stddevNorm']
chart_helper.draw_violin(df, features_to_plot, dict_group = four_group_names)

In [ ]:
import pandas as pd

# ----------------------------------------------------------
# Complete eGeMAPSv02 feature list (88 features)
# ----------------------------------------------------------
egemaps_features = {
    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "F0semitoneFrom27.5Hz_sma3nz_stddevNorm",
    "F0semitoneFrom27.5Hz_sma3nz_percentile20.0",
    "F0semitoneFrom27.5Hz_sma3nz_percentile50.0",
    "F0semitoneFrom27.5Hz_sma3nz_percentile80.0",
    "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    "F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope",
    "F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope",

    "loudness_sma3_amean",
    "loudness_sma3_stddevNorm",
    "loudness_sma3_percentile20.0",
    "loudness_sma3_percentile50.0",
    "loudness_sma3_percentile80.0",
    "loudness_sma3_pctlrange0-2",
    "loudness_sma3_meanRisingSlope",
    "loudness_sma3_stddevRisingSlope",
    "loudness_sma3_meanFallingSlope",
    "loudness_sma3_stddevFallingSlope",

    "spectralFlux_sma3_amean",
    "spectralFlux_sma3_stddevNorm",

    "mfcc1_sma3_amean",
    "mfcc1_sma3_stddevNorm",
    "mfcc2_sma3_amean",
    "mfcc2_sma3_stddevNorm",
    "mfcc3_sma3_amean",
    "mfcc3_sma3_stddevNorm",
    "mfcc4_sma3_amean",
    "mfcc4_sma3_stddevNorm",

    "jitterLocal_sma3nz_amean",
    "jitterLocal_sma3nz_stddevNorm",

    "shimmerLocaldB_sma3nz_amean",
    "shimmerLocaldB_sma3nz_stddevNorm",

    "HNRdBACF_sma3nz_amean",
    "HNRdBACF_sma3nz_stddevNorm",

    "alphaRatioV_sma3nz_amean",
    "alphaRatioV_sma3nz_stddevNorm",

    "hammarbergIndexV_sma3nz_amean",
    "hammarbergIndexV_sma3nz_stddevNorm",

    "slopeV0-500_sma3nz_amean",
    "slopeV0-500_sma3nz_stddevNorm",

    "slopeV500-1500_sma3nz_amean",
    "slopeV500-1500_sma3nz_stddevNorm",

    "F1frequency_sma3nz_amean",
    "F1frequency_sma3nz_stddevNorm",
    "F2frequency_sma3nz_amean",
    "F2frequency_sma3nz_stddevNorm",
    "F3frequency_sma3nz_amean",
    "F3frequency_sma3nz_stddevNorm",

    "F1bandwidth_sma3nz_amean",
    "F1bandwidth_sma3nz_stddevNorm",
    "F2bandwidth_sma3nz_amean",
    "F2bandwidth_sma3nz_stddevNorm",
    "F3bandwidth_sma3nz_amean",
    "F3bandwidth_sma3nz_stddevNorm",

    "logRelF0-H1-H2_sma3nz_amean",
    "logRelF0-H1-H2_sma3nz_stddevNorm",

    "logRelF0-H1-A3_sma3nz_amean",
    "logRelF0-H1-A3_sma3nz_stddevNorm",

    "F0env_sma3nz_amean",
    "F0env_sma3nz_stddevNorm"
}

# ----------------------------------------------------------
# Filter dataframe
# Keep:
# 1. First two columns
# 2. Columns that contain any eGeMAPS feature name
# ----------------------------------------------------------

keep_columns = list(df.columns[:2])

for col in df.columns[2:]:

    # As long as the field name contains any eGeMAPS feature
    if any(feature in col for feature in egemaps_features):
        keep_columns.append(col)

df_accoustics = df[keep_columns].copy()

print("Original shape:", df.shape)
print("Filtered shape:", df_accoustics.shape)

In [ ]:
list(df_accoustics.columns)

### Run Analysis

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df_accoustics, df_name, 
    param_combinations,  experiment_name = "Experiment 1: Accoustic Features",
    parameters = parameters,
    random_state = random_state,
    remove_high_correlation = True)

print(df_all_evaluations)
print(df_all_predictions)

df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}

# ================================
# Execute classification analysis task (Main Task)
# ================================

# emotion_classification_task is a custom task module,
# The run_analysis() method is responsible for executing the entire sentiment classification analysis pipeline.
# It typically includes the following steps:
# 1. Execute feature selection (ANOVA or RFE) based on parameter combinations;
# 2. Train using the specified classification model (e.g. Random Forest);
# 3. If Grid Search is enabled, automatically tune parameters;
# 4. Calculate model performance metrics (accuracy, precision, recall, F1-score, etc.);
# 5. Return results and predicted values.
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df_accoustics,             # input dataset (DataFrame)
    df_name,        # Dataset name (for logging or records)
    param_combinations,  # List of all parameter combinations
    experiment_name = "Experiment 1: Accoustic Features",
    random_state = random_state,
    remove_high_correlation = False
)


# ================================
# Output analysis results (Results Output)
# ================================

# Print all model evaluation results (e.g., accuracy, F1 score for each parameter group)
print(df_all_evaluations)

# Print prediction results for all samples (predicted class for each sample)
print(df_all_predictions)


# ================================
# Merge multimodal results (Multimodal Integration)
# ================================

# If the experiment contains multiple modalities (e.g., text, image, audio),
# Results DataFrames from different modalities can be merged into a summary DataFrame for convenient downstream statistical analysis.
# pd.concat() is used to concatenate multiple DataFrames.
# ignore_index=True means reindex the index after merging.
df_multimodal_evaluations = pd.concat(
    [df_multimodal_evaluations, df_all_evaluations],
    ignore_index=True
)

df_multimodal_predictions = pd.concat(
    [df_multimodal_predictions, df_all_predictions],
    ignore_index=True
)


In [ ]:
# # Set a fixed random state for reproducibility
# # This ensures that the results are consistent across runs by initializing the random seed
# random_state = 1

# group_column_list = ["user_id"]

# # Define parameter options
# # Number of features to retain after feature selection
# n_kept_list = [15]  # This can be adjusted to include more values for experimentation

# # Boolean flags indicating whether to perform specific preprocessing steps
# # Whether to perform ANOVA (Analysis of Variance) feature selection

# do_anova_list = [True]  # ANOVA helps in selecting features with significant differences across classes

# # Whether to perform Recursive Feature Elimination (RFE), a wrapper method for feature selection
# do_RFE_list = [False]

# # Whether to perform model hyperparameter tuning using Grid Search
# do_model_grid_search_list = [False]  # Grid Search is computationally expensive but optimizes hyperparameters

# # List of models to use for RFE (Recursive Feature Elimination)
# RFE_model_name_list = ["SVM"]  # Options can include "Decision Tree", "XGBoost", etc.

# # List of models to use for the final classification task
# default_model_name_list = ["Random Forest"]  # Options can include "SVM", "Logistic Regression", etc.

# # Generate all combinations of parameter values using Cartesian product
# # This ensures we test all combinations of the hyperparameters defined above
# param_combinations = list(itertools.product(
#     n_kept_list, do_anova_list, do_RFE_list, do_model_grid_search_list, default_model_name_list, RFE_model_name_list, group_column_list
# ))

# df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(df, df_name, param_combinations)

# print(df_all_evaluations)
# print(df_all_predictions)

# df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
# df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   


## 2. Speech Text

### Definition

In [ ]:
df = all_df[1].copy()
df_name = "Speech Text"

# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.
number_of_features = 15  # Fixed number of retained features (matches Methods)
n_kept_list = [number_of_features]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [True]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]
# RFE_model_name_list = ["Random Forest"]

# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]
# default_model_name_list = ["Random Forest"]

# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

In [ ]:
features_to_plot = ['median_speech_text-surprise']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 3
chart_helper.draw_violin(df = df, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)

In [ ]:
list(df.columns)

In [ ]:
keep_columns = list(df.columns[:2])
keep_columns

In [ ]:
# ----------------------------------------------------------
# Filter dataframe
# Keep:
# 1. First two columns
# 2. Columns that match: median_acoustics-[eGeMAPS feature]
# ----------------------------------------------------------
keep_columns = list(df.columns[:2])

for col in df.columns[2:]:

    if not col.startswith("median_speech_text-"):
        continue

    feature = col.split("median_speech_text-")[1]
    keep_columns.append(col)
       

df_speech_text = df[keep_columns].copy()

print("Filtered shape:", df_speech_text.shape)

In [ ]:
df_speech_text

### Run Anaysis

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df_speech_text, df_name, 
    param_combinations,  experiment_name = "Experiment 1: Speech Text Emotion",
    parameters = parameters,
    random_state = random_state,
    remove_high_correlation = True)

print(df_all_evaluations)
print(df_all_predictions)

df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

In [ ]:
# df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(df, df_name, param_combinations, experiment_name = "Experiment 1: Speech Text Emotion")

# print(df_all_evaluations)
# print(df_all_predictions)

# df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
# df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

In [ ]:
print(df_all_evaluations)

## 3. Facial Expression 

In [ ]:
df = all_df[2].copy()
df_name = "Facial"

# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.
number_of_features = 15  # Fixed number of retained features (matches Methods)
n_kept_list = [number_of_features]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [True]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]
# RFE_model_name_list = ["Random Forest"]

# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]
# default_model_name_list = ["Random Forest"]

# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

In [ ]:
features_to_plot = ['median_AU12']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 3
chart_helper.draw_violin(df = df, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)
df_facial = df.copy()

In [ ]:
list(df.columns)

In [ ]:
# keep_columns = list(df.columns[:2])

# for col in df.columns[2:]:

# Keep only entries beginning with end_to_begin_
#     if not col.startswith("end_to_begin_"):
#         continue

#     feature = col.split("end_to_begin_")[1]

# Exclude fields containing Pitch / Roll / Yaw
#     if any(k in feature for k in ["Pitch", "Roll", "Yaw"]):
#         continue

#     keep_columns.append(col)

# df_trimmed = df[keep_columns].copy()
# df_trimmed

# Fields to be added
# cols_to_add = ["user_id", "variance_Pitch", "variance_Roll", "variance_Yaw"]

# Extract the required fields
# df_pitch = df[cols_to_add].copy()


In [ ]:
# ----------------------------------------------------------
# Identify columns containing Pitch / Roll / Yaw / FaceRect
# ----------------------------------------------------------
keywords = ["Pitch", "Roll", "Yaw", "FaceRect"]

pitch_cols = [c for c in df_facial.columns if any(k in c for k in keywords)]

# ----------------------------------------------------------
# Create df_pitch (keep user_id + these columns)
# ----------------------------------------------------------
df_pitch = df_facial[["user_id"] + pitch_cols].copy()

# ----------------------------------------------------------
# Remove them from df_facial (but keep user_id)
# ----------------------------------------------------------
df_facial = df_facial.drop(columns=pitch_cols)

print("Facial shape:", df_facial.shape)
print("Pitch/Roll/Yaw shape:", df_pitch.shape)

### Run Anaysis

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df_facial, df_name, 
    param_combinations,  experiment_name = "Experiment 1: Facial Expression",
    parameters = parameters,
    random_state = random_state,
    remove_high_correlation = True)

print(df_all_evaluations)
print(df_all_predictions)

df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

In [ ]:
# df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(df_facial, df_name, param_combinations, experiment_name = "Experiment 1: Facial Expression")

# print(df_all_evaluations)
# print(df_all_predictions)

# df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
# df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

## 4. Pose Analysis

### Definition

In [ ]:
df = all_df[3].copy()
df_name = "pose"

# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.
# number_of_features = max(15, len(df_pose.columns) // 5)
number_of_features = 15  # Fixed number of retained features (matches Methods)
n_kept_list = [number_of_features]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [True]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]
# RFE_model_name_list = ["Random Forest"]

# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]
# default_model_name_list = ["Random Forest"]

# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

In [ ]:
# df = all_df[3]
# df_name = "pose"
features_to_plot = ['pose-max_left_head_angle']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 3
chart_helper.draw_violin(df = df, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)

In [ ]:
# list(df.columns)

In [ ]:
keep_columns = list(df.columns[:2])

for col in df.columns[2:]:

    key_words = "pose-variance_"
    # Keep only lines starting with key_words
    if not col.startswith(key_words):
        continue

    feature = col.split(key_words)[1]

    # Exclude fields containing Pitch / Roll / Yaw
    if any(k in feature for k in ["Pitch", "Roll", "Yaw"]):
        continue

    keep_columns.append(col)

df_trimmed = df[keep_columns].copy()
df_trimmed

In [ ]:
df_head_pose = df_trimmed.merge(df_pitch, on="user_id", how="left")
df_head_pose

In [ ]:
features_to_plot = ['end_to_begin_Yaw']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 3
chart_helper.draw_violin(df = df_head_pose, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)

In [ ]:
features_to_plot = ['variance_FaceRectWidth']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 3
chart_helper.draw_violin(df = df_head_pose, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)

In [ ]:
df_head_pose

In [ ]:
# Identify rows that contain NaN
rows_with_na = df_head_pose[df_head_pose.isna().any(axis=1)]

# Print removed rows
print("Rows to be removed:")
print(rows_with_na)

# Remove them
df_head_pose_clean = df_head_pose.dropna()

# Optional: show how many rows were removed
print(f"\nRemoved {len(rows_with_na)} rows.")
print(f"Original shape: {df_head_pose.shape}")
print(f"New shape: {df_head_pose_clean.shape}")

### Run Anaysis

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df_head_pose_clean, df_name, 
    param_combinations,  experiment_name = "Experiment 1: Head Position",
    parameters = parameters,
    random_state = random_state,
    remove_high_correlation = True)

print(df_all_evaluations)
print(df_all_predictions)

df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

### Run Anaysis

In [ ]:
# df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(df_head_pose_clean, df_name, param_combinations, experiment_name = "Experiment 1: Head Position Analysis")

# print(df_all_evaluations)
# print(df_all_predictions)

# df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
# df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

## 5. Eye Tracking

### Definition

In [ ]:
df = all_df[4].copy()
df_name = "Eye_Movement"

# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.
# number_of_features = max(10, len(df.columns) // 5)
number_of_features = 15  # Fixed number of retained features (matches Methods)
n_kept_list = [number_of_features]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [True]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]
# RFE_model_name_list = ["Random Forest"]

# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]
# default_model_name_list = ["Random Forest"]

# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

In [ ]:
features_to_plot = ['max_eye_move-mean_right_iris_relative_x']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 3
chart_helper.draw_violin(df = df, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)


In [ ]:
list(df.columns)

In [ ]:
keep_columns = list(df.columns[:2])

for col in df.columns[2:]:

    key_words = "median_eye_move-"
    # Keep only lines starting with key_words
    if not col.startswith(key_words):
        continue

    feature = col.split(key_words)[1]


    keep_columns.append(col)

df_eye_movement = df[keep_columns].copy()
df_eye_movement

### Run Anaysis

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df_eye_movement, df_name, 
    param_combinations,  experiment_name = "Experiment 1: Eye Movement",
    parameters = parameters,
    random_state = random_state,
    remove_high_correlation = True)

print(df_all_evaluations)
print(df_all_predictions)

df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

In [ ]:
# df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
#     df_eye_movement, df_name, 
#     param_combinations, 
#     experiment_name = "Experiment 1: Eye Movement",
#     )

# print(df_all_evaluations)
# print(df_all_predictions)

# df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
# df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

## 6. Remote PPG

In [ ]:
df = all_df[5]
df_name = "r-PPG"

# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.
# number_of_features = max(10, len(df.columns) // 5)
number_of_features = 15  # Fixed number of retained features (matches Methods)
n_kept_list = [number_of_features]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [True]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]
# RFE_model_name_list = ["Random Forest"]

# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]
# default_model_name_list = ["Random Forest"]

# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

In [ ]:

features_to_plot = ['median_r_ppg-rMSSD']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 3
chart_helper.draw_violin(df = df, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)

In [ ]:
list(df.columns)

### Run Anaysis

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df, df_name, 
    param_combinations,  experiment_name = "Experiment 1: Remote PPG",
    parameters = parameters,
    random_state = random_state,
    remove_high_correlation = True)

print(df_all_evaluations)
print(df_all_predictions)

df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

In [ ]:
# df_ppg  = df.copy()
# df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(df_ppg, df_name, param_combinations, experiment_name = "Experiment 1: Remote PPG" )

# print(df_all_evaluations)
# print(df_all_predictions)

# df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
# df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

## 7. Thermal

### Definition

In [ ]:
df = all_df[6].copy()
df_name = "thermal"

# ================================
# 1. Setting random seed (Random State)
# ================================

# Set fixed random seed to ensure reproducibility of results.
# In machine learning experiments, model training typically involves stochastic processes (e.g., data splitting, parameter initialization, tree sampling in random forests, etc.).
# If the random seed is not set, model results may vary slightly with each run.
# Therefore, to facilitate experimental comparison or paper reproduction, a random number seed is typically fixed.
random_state = 42


# ================================
# 2. Defining Group Columns
# ================================

# Specify the column name for grouping or cross-validation.
# Example: In emotion recognition experiments, group by "user_id" to ensure that data from the same user does not appear in both training and test sets simultaneously.
# This prevents data leakage and ensures model generalization capability.
group_column_list = ["user_id"]


# ================================
# III. Define Parameter Options
# ================================

# 1️⃣ Number of features retained (n_kept_list)
# Indicates the number of features to retain after feature selection.
# Can set multiple values for experimenting with performance differences across different feature dimensions.

number_of_features = 15  # Fixed number of retained features (matches Methods)
n_kept_list = [number_of_features]  # Example retains 15 features; can be changed to [10, 20, 30] etc. for hyperparameter search


# 2️⃣ Whether to execute ANOVA feature selection (do_anova_list)
# ANOVA (Analysis of Variance) is a statistical method
# Can help filter out features with significant differences between different categories.
# For classification tasks (such as sentiment recognition), using ANOVA can improve feature effectiveness.
do_anova_list = [False]


# 3️⃣ Should RFE (Recursive Feature Elimination) be executed
# RFE is a model-based feature selection method.
# It trains a base model (e.g., SVM or decision tree), then recursively removes the least important features,
# Until retained to the specified quantity.
# This location is True, indicating that this method is enabled.
do_RFE_list = [True]


# 4️⃣ Whether to perform model hyperparameter tuning (Grid Search)
# Grid Search is a hyperparameter optimization method.
# It traverses preset parameter combinations to find the parameter configuration that optimizes model performance.
# This process is computationally intensive, so it can be set to False initially and enabled in subsequent experiments.
do_model_grid_search_list = [False]


# 5️⃣ Model type list for RFE (RFE_model_name_list)
# Specify the base model type used when executing RFE.
# For example, you can choose SVM, Decision Tree, XGBoost, etc.
RFE_model_name_list = ["SVM"]
# RFE_model_name_list = ["Random Forest"]

# 6️⃣ Final classification model type list (default_model_name_list)
# Specify the model to use for the final classification task.
# For example, you can choose Random Forest, Support Vector Machine (SVM),
# Logistic Regression, etc.
default_model_name_list = ["Random Forest"]
#default_model_name_list = ["Random Forest"]

# ================================
# 4. Generate all parameter combinations (Cartesian Product)
# ================================

# Using itertools.product() to generate Cartesian product of all parameters.
# That is, it will combine all values in the parameter list pairwise.
# So that subsequent systematic testing of model performance across each parameter configuration can proceed.
# For example, if n_kept_list has two values, do_anova_list has two values, then the number of combinations is 2×2×...
param_combinations = list(itertools.product(
    n_kept_list,                 # Feature retention count
    do_anova_list,               # Whether to execute ANOVA
    do_RFE_list,                 # Whether to execute RFE
    do_model_grid_search_list,   # Whether to execute Grid Search
    default_model_name_list,     # Final model type
    RFE_model_name_list,         # RFE uses model type
    group_column_list            # group column
))

# Example output:
# [(15, True, False, False, 'Random Forest', 'SVM', 'user_id')]
# Each tuple represents a set of experimental parameter configurations.

In [ ]:

features_to_plot = ['thermal-variance_thermal_feature_11']
to_remove_outlier = False
do_within_group_t_test = True
do_between_group_t_test = True
outlier_multiplier = 1.5
chart_helper.draw_violin(df = df, 
                         features_to_plot = features_to_plot, 
                         outlier_multiplier = outlier_multiplier, 
                         to_remove_outlier = to_remove_outlier, 
                         do_between_group_t_test = do_between_group_t_test,
                         do_within_group_t_test = do_within_group_t_test, 
                         dict_group = four_group_names)

In [ ]:
parameters = {
    "run_1": False,  # First experimental run
    "run_2": False,  # Second experimental run
    "run_3": True,  # Third experimental run
    "verbose": True,  # Whether to print detailed logs during training
    "n_cross_validation": 5,  # Number of folds for cross-validation
}
df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
    df, df_name, 
    param_combinations,  experiment_name = "Experiment 1: Thermal Imaging",
    parameters = parameters,
    random_state = random_state,
    remove_high_correlation = True)

print(df_all_evaluations)
print(df_all_predictions)

df_multimodal_evaluations = pd.concat([df_multimodal_evaluations, df_all_evaluations], ignore_index=True)
df_multimodal_predictions = pd.concat([df_multimodal_predictions, df_all_predictions], ignore_index=True)   

In [ ]:
df_multimodal_predictions

## 8. MLLM

In [ ]:
# experiment_helper.define_experiment(
#     participant_sn_range= list_users,
#     path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", f"merged_video_23.plk"),
#     path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", f"df_23_exp_2.plk"),
#     path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
#     experiment_step=step,
#     video_only=True,
#     template_main= "",
#     interpreters = [],
#     template_cot = "",
#     template_knowledge_injection= "",
#     template_tools= "",
# )
# df_result_23_2 = experiment_helper.get_df_experiment_results()
# experiment_helper.define_experiment(
#     participant_sn_range= list_users,
#     path_pickle_df_video= os.path.join(path_paper, "temp", "pickle", f"merged_video_25.plk"),
#     path_pickle_df_experiment_results=os.path.join(path_paper, "temp", "pickle", f"df_25_exp_2.plk"),
#     path_compressed_video= os.path.join(path_paper, "temp", "compressed_video"),
#     experiment_step= step,
#     video_only=False,
#     interpreters=[],
#     template_main= "",
#     template_cot = "",
#     template_knowledge_injection= "",
#     template_tools= ""
# )
# df_result_25_2 = experiment_helper.get_df_experiment_results()

In [ ]:
# with pd.option_context("display.max_colwidth", None):
#     display(df_result_23_2[["Group Name","sympathy","inspiration","LLM_Result"]])

In [ ]:
# # -------------------------------------------------------
# # 1. Merge on identification column "Participant SN"
# # -------------------------------------------------------
# df_merged = pd.merge(
#     df_result_23_2,
#     df_result_25_2[["Participant SN", "explanation"]],
#     on="Participant SN",
#     how="inner",
#     suffixes=("_df1", "_df2")
# )

# # -------------------------------------------------------
# # 2. Concatenate explanation columns
# # -------------------------------------------------------
# df_merged["decoded_text"] = (
#     df_merged["explanation_df1"].fillna("").astype(str) + " " +
#     df_merged["explanation_df2"].fillna("").astype(str)
# ).str.strip()

# # -------------------------------------------------------
# # 3. Keep required columns and rename
# # -------------------------------------------------------
# df_merged = df_merged[[
#     "Participant_SN_int",
#     "Participant SN",
#     "Group",
#     "decoded_text"
# ]].rename(columns={
#     "Participant_SN_int": "user_id"
# })


# Only retain English letters (A–Z, a–z), remove everything else
# df_merged["decoded_text"] = (
#     df_merged["decoded_text"]
#     .fillna("")
#     .astype(str)
#     .apply(lambda x: re.sub(r"[^A-Za-z\s]", "", x))
# )

# # 1. Compress excess whitespace into single spaces
# # 2. Remove leading and trailing whitespace
# df_merged["decoded_text"] = (
#     df_merged["decoded_text"]
#     .apply(lambda x: re.sub(r"\s+", " ", x).strip())
# )


# # Initialize stopwords and lemmatizer
# from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer
# from nltk.tokenize import word_tokenize
# import string


# stop_words = set(stopwords.words('english'))
# lemmatizer = WordNetLemmatizer()

# # Preprocessing function
# def preprocess_text(text):
#     # Lowercase
#     text = text.lower()

#     # remove problematic words
#     text = text.replace("<|eot_id|>", "", 1)
    
#     # Remove punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))
    
#     # Tokenize
#     tokens = word_tokenize(text)
    
#     # Remove stopwords and lemmatize
#     tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
#     return ' '.join(tokens)

# # Apply preprocessing to the text data
# df_merged['Processed_Text'] = df_merged['decoded_text'].apply(preprocess_text)


# # --------------------------------------------
# # If first use please uncomment below to download
# # --------------------------------------------
# # nltk.download("wordnet")
# # nltk.download("omw-1.4")

# lemmatizer = WordNetLemmatizer()

# # --------------------------------------------
# # Custom text preprocessing + lemmatization
# # --------------------------------------------
# def custom_analyzer(text):
# 1. Convert to lowercase
#     text = text.lower()
    
# 2. Keep only English letters and whitespace
#     text = re.sub(r"[^a-z\s]", "", text)
    
# 3. Word segmentation
#     tokens = text.split()
    
#     # 4. Lemmatize
#     tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
#     return tokens

# # --------------------------------------------
# Initialize improved TF-IDF
# # --------------------------------------------
# tfidf = TfidfVectorizer(
#     analyzer=custom_analyzer,
#     stop_words="english",
#     ngram_range=(1, 2),     # unigram + bigram
#     max_features=3000,      # Can adjust based on data volume
# sublinear_tf=True       # More stable for long texts
# )

# # --------------------------------------------
# # Convert to TF-IDF vector
# # --------------------------------------------
# fitted_tfidf = tfidf.fit_transform(
#     df_merged["Processed_Text"].fillna("").astype(str)
# )

# # --------------------------------------------
# Convert to DataFrame (for easy viewing)
# # --------------------------------------------
# tfidf_df = pd.DataFrame(
#     fitted_tfidf.toarray(),
#     columns=tfidf.get_feature_names_out()
# )

# # Step 4: Concatenate the TF-IDF DataFrame with the original df_read
# df_merged_tfidf = pd.concat([df_merged.reset_index(drop=True), tfidf_df], axis=1)

# # rename
# df_merged_tfidf_renamed = df_merged_tfidf.rename(
#     columns={
#         col: f"mm_llm-{col}"
#         for col in df_merged_tfidf.columns[5:]   # Starting from column 5 (index=4)
#     }
# )


# Get all column names after the fifth column (index=4)
# cols_from_5th = df_merged_tfidf_renamed.columns[5:]


# Establish reserved fields list
# cols_to_keep = ["user_id", "Group"] + list(cols_from_5th)

# # Keep only specified fields
# df_filtered = df_merged_tfidf_renamed[cols_to_keep].copy()

# df_all_evaluations, df_all_predictions = emotion_classification_task.run_analysis(
# df = df_filtered,             # input dataset (DataFrame)
# df_name = "MLLM",        # dataset name (used for logging or records)
# param_combinations = param_combinations,  # all parameter combination list
#     experiment_name = "Experiment 1: MLLM"
# )


# Late Fusion

## Experiments 10-13: Late Fusion (majority vote)

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics import accuracy_score, confusion_matrix


# Only MajorityVotingFusion is used for the reported Exp 10-13 results (Appendix C.2.4); the other fusion classes are retained but not reported.
class LateFusionBase:
    """
    Base class for multimodal late fusion strategies.
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def fuse(self):
        raise NotImplementedError("Subclasses must implement fuse()")

    # def evaluate(self, df_fused):
    #     """
    #     Evaluate accuracy and confusion matrix.
    #     """

    #     acc = accuracy_score(df_fused["actual"], df_fused["predicted_fused"])
    #     cm = confusion_matrix(df_fused["actual"], df_fused["predicted_fused"])

    #     return acc, cm

    def evaluate(self, df_fused, average="weighted"):
        """
        Evaluate prediction performance.

        Parameters
        ----------
        df_fused : pd.DataFrame
            DataFrame containing columns:
            - actual
            - predicted_fused

        average : str
            F1 averaging method ("binary", "macro", "micro", "weighted").
            Default = "weighted", recommended for multiclass with imbalance.

        Returns
        -------
        acc : float
            Accuracy score

        f1 : float
            F1 score

        cm : np.ndarray
            Confusion matrix
        """

        y_true = df_fused["actual"]
        y_pred = df_fused["predicted_fused"]

        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average=average)
        cm = confusion_matrix(y_true, y_pred)

        return acc, f1, cm

class MajorityVotingFusion(LateFusionBase):

    def fuse(self):

        results = []

        for user_id, group in self.df.groupby("user_id"):

            fused_pred = group["predicted"].mode().iloc[0]
            actual = group["actual"].iloc[0]

            results.append({
                "user_id": user_id,
                "predicted_fused": fused_pred,
                "actual": actual
            })

        df_fused = pd.DataFrame(results)

        return df_fused

class WeightedVotingFusion(LateFusionBase):

    def __init__(self, df, weights):
        super().__init__(df)
        self.weights = weights

    def fuse(self):

        results = []

        for user_id, group in self.df.groupby("user_id"):

            score_dict = defaultdict(float)

            for _, row in group.iterrows():

                modality = row["modality"]
                pred = row["predicted"]

                weight = self.weights.get(modality, 1)

                score_dict[pred] += weight

            fused_pred = max(score_dict, key=score_dict.get)
            actual = group["actual"].iloc[0]

            results.append({
                "user_id": user_id,
                "predicted_fused": fused_pred,
                "actual": actual
            })

        return pd.DataFrame(results)

# class MeanFusion(LateFusionBase):

#     def fuse(self):

#         df_fused = (
#             self.df.groupby("user_id")
#             .agg(
#                 predicted_fused=("predicted", "mean"),
#                 actual=("actual", "first")
#             )
#             .reset_index()
#         )

#         return df_fused

class MeanFusion(LateFusionBase):

    def fuse(self):

        df_pivot = self.df.pivot_table(
            index="user_id",
            columns="modality",
            values="predicted"
        )

        # mean across modalities
        df_pivot["predicted_fused"] = df_pivot.mean(axis=1)

        df_actual = self.df.groupby("user_id")["actual"].first()

        df_fused = pd.DataFrame({
            "user_id": df_pivot.index,
            "predicted_fused": df_pivot["predicted_fused"].round().astype(int),
            "actual": df_actual
        }).reset_index(drop=True)

        return df_fused

class MaxScoreFusion(LateFusionBase):

    def fuse(self):

        results = []

        for user_id, group in self.df.groupby("user_id"):

            row = group.loc[group["predicted"].idxmax()]

            results.append({
                "user_id": user_id,
                "predicted_fused": row["predicted"],
                "actual": row["actual"]
            })

        return pd.DataFrame(results)

from sklearn.linear_model import LogisticRegression

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


class StackingFusion(LateFusionBase):

    def __init__(
        self,
        df,
        model=None,
        n_splits=5,
        impute_strategy="most_frequent"
    ):

        super().__init__(df)

        if model is None:
            model = LogisticRegression(max_iter=1000)

        self.model = model
        self.n_splits = n_splits
        self.impute_strategy = impute_strategy


    def _pivot_modalities(self):

        df_pivot = self.df.pivot_table(
            index="user_id",
            columns="modality",
            values="predicted",
            aggfunc="first"
        )

        df_actual = self.df.groupby("user_id")["actual"].first()

        return df_pivot, df_actual


    def fuse(self):

        df_pivot, y = self._pivot_modalities()

        X = df_pivot.copy()
        groups = df_pivot.index

        imputer = SimpleImputer(strategy=self.impute_strategy)
        X = imputer.fit_transform(X)

        gkf = GroupKFold(n_splits=self.n_splits)

        preds_all = []
        user_ids = []

        for train_idx, test_idx in gkf.split(X, y, groups):

            X_train, X_test = X[train_idx], X[test_idx]
            y_train = y.iloc[train_idx]

            model = self.model
            model.fit(X_train, y_train)

            preds = model.predict(X_test)

            preds_all.extend(preds)
            user_ids.extend(groups[test_idx])

        df_fused = pd.DataFrame({
            "user_id": user_ids,
            "predicted_fused": preds_all
        })

        df_fused = df_fused.merge(
            y.rename("actual"),
            left_on="user_id",
            right_index=True
        )

        return df_fused

import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC


class StackingFusion(LateFusionBase):

    def __init__(
        self,
        df,
        model_type="logistic",
        model_params=None,
        n_splits=5,
        impute_strategy="most_frequent"
    ):
        """
        Parameters
        ----------
        df : DataFrame
            dataframe with columns: modality, predicted, actual, user_id

        model_type : str
            meta-classifier type

        model_params : dict
            parameters for the chosen model

        n_splits : int
            number of GroupKFold splits

        impute_strategy : str
            strategy for missing modalities
        """

        super().__init__(df)

        if model_params is None:
            model_params = {}

        self.model_type = model_type
        self.model_params = model_params
        self.n_splits = n_splits
        self.impute_strategy = impute_strategy

        self.model = self._build_model()

    def _build_model(self):

        if self.model_type == "logistic":
            return LogisticRegression(max_iter=1000, **self.model_params)

        elif self.model_type == "random_forest":
            return RandomForestClassifier(**self.model_params)

        elif self.model_type == "gradient_boosting":
            return GradientBoostingClassifier(**self.model_params)

        elif self.model_type == "svm":
            return SVC(probability=True, **self.model_params)

        elif self.model_type == "xgboost":
            from xgboost import XGBClassifier
            return XGBClassifier(**self.model_params)

        elif self.model_type == "lightgbm":
            from lightgbm import LGBMClassifier
            return LGBMClassifier(**self.model_params)

        else:
            raise ValueError("Unknown model_type")

    def _pivot_modalities(self):

        df_pivot = self.df.pivot_table(
            index="user_id",
            columns="modality",
            values="predicted",
            aggfunc="first"
        )

        y = self.df.groupby("user_id")["actual"].first()

        return df_pivot, y

    def fuse(self):

        df_pivot, y = self._pivot_modalities()

        X = df_pivot.copy()
        groups = df_pivot.index

        # handle missing modalities
        imputer = SimpleImputer(strategy=self.impute_strategy)
        X = imputer.fit_transform(X)

        gkf = GroupKFold(n_splits=self.n_splits)

        preds_all = []
        user_ids = []

        for train_idx, test_idx in gkf.split(X, y, groups):

            X_train, X_test = X[train_idx], X[test_idx]
            y_train = y.iloc[train_idx]

            model = self._build_model()
            model.fit(X_train, y_train)

            preds = model.predict(X_test)

            preds_all.extend(preds)
            user_ids.extend(groups[test_idx])

        df_fused = pd.DataFrame({
            "user_id": user_ids,
            "predicted_fused": preds_all
        })

        df_fused = df_fused.merge(
            y.rename("actual"),
            left_on="user_id",
            right_index=True
        )

        return df_fused

In [ ]:
df_multimodal_predictions.rename(columns={"all_group": "user_id"}, inplace=True)
df_multimodal_predictions.rename(columns={"all_y_true": "actual"}, inplace=True)
df_multimodal_predictions.rename(columns={"all_y_pred": "predicted"}, inplace=True)
df_multimodal_predictions.rename(columns={"df_name": "modality"}, inplace=True)
df_prepared = df_multimodal_predictions.copy()
df_prepared.to_csv("machine_learning.csv")
df_prepared


In [ ]:
df_prepared["modality"].unique()

In [ ]:
df_name_part_1 = ['Facial', 'pose', 'Eye_Movement']
df_part_1 = df_prepared[
    df_prepared["modality"].isin(df_name_part_1)
]

df_name_part_2 = ['Accoustics', 'Facial', 'pose', 'Eye_Movement']
df_part_2 = df_prepared[
    df_prepared["modality"].isin(df_name_part_2)
]

df_name_part_3 = ['Accoustics', 'Speech Text', 'Facial', 'pose', 'Eye_Movement']
df_part_3 = df_prepared[
    df_prepared["modality"].isin(df_name_part_3)
]

df_name_part_4 = ['Accoustics', 'Speech Text', 'Facial', 'pose', 'Eye_Movement',  'r-PPG', 'thermal']
df_part_4 = df_prepared[
    df_prepared["modality"].isin(df_name_part_4)
]

In [ ]:
group_names = {1: "ELVT", 2: "SYMP", 3: "INSP", 4: "CTRL"}
group_labels = [group_names[i] for i in [1, 2, 3, 4]]

# Majority voting
fusion = MajorityVotingFusion(df_part_1)
df_fused = fusion.fuse()
acc, f1, cm = fusion.evaluate(df_fused)

# Trial 2

import numpy as np

group_names = {1: "ELVT", 2: "SYMP", 3: "INSP", 4: "CTRL"}
group_labels = [group_names[i] for i in [1, 2, 3, 4]]

# Majority voting
fusion = MajorityVotingFusion(df_part_1)
df_fused = fusion.fuse()
acc, f1, cm = fusion.evaluate(df_fused)
df_fused.to_csv("experiment_10.csv")

# Total count for each class (true class)
row_sums = cm.sum(axis=1)

values = []

print("\nPer-class accuracy (correct / total):")
for i in range(len(group_labels)):
    correct = cm[i, i]
    total = row_sums[i]

    ratio = correct / total if total != 0 else 0

    print(f"{group_labels[i]}: {correct}/{total} = {ratio:.2%}")

    values.append(f"{ratio:.4f}")  # For Excel use

# Excel one column (tab-separated)
excel_row = "\t".join(values)
print("\nExcel (row):")
print(excel_row)

# Trial 1
# Normalize each column (avoid division by zero)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(cm, row_sums, where=row_sums != 0)

# Get diagonal (accuracy for each class)
per_class_accuracy = np.diag(cm_normalized)

# # Corresponding label
# for i, acc_i in enumerate(per_class_accuracy):
#     print(f"Class {group_labels[i]} accuracy: {acc_i:.2%}")

# Corresponding label
values = []
for i, acc_i in enumerate(per_class_accuracy):
    print(f"Class {group_labels[i]} accuracy: {acc_i:.2%}")
    values.append(f"{acc_i:.4f}")  # or use .2% depending on your needs

# Excel one column (tab-separated)
excel_row = "\t".join(values)
print("\nExcel (row):")
print(excel_row)

title = "Exp 10. V"
chart_helper.plot_confusion_matrix(
    cm,
    ticklabels=group_labels,
    show_percentage=True,
    task_name="",
    title=f"{title} (Acc:{acc:.2f}, F1:{f1:.2f})",
    xticklabels=group_labels,
    yticklabels=group_labels,
    font_size = 20
)

# Majority voting
fusion = MajorityVotingFusion(df_part_2)
df_fused = fusion.fuse()
acc, f1, cm = fusion.evaluate(df_fused)
df_fused.to_csv("experiment_11.csv")

# Normalize each column (avoid division by zero)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(cm, row_sums, where=row_sums != 0)

# Get diagonal (accuracy for each class)
per_class_accuracy = np.diag(cm_normalized)

# # Corresponding label
# for i, acc_i in enumerate(per_class_accuracy):
#     print(f"Class {group_labels[i]} accuracy: {acc_i:.2%}")

# Corresponding label
values = []
for i, acc_i in enumerate(per_class_accuracy):
    print(f"Class {group_labels[i]} accuracy: {acc_i:.2%}")
    values.append(f"{acc_i:.4f}")  # or use .2% depending on your needs

# Excel one column (tab-separated)
excel_row = "\t".join(values)
print("\nExcel (row):")
print(excel_row)

title = "Exp 11. V+A"
chart_helper.plot_confusion_matrix(
    cm,
    ticklabels=group_labels,
    show_percentage=True,
    task_name="",
    title=f"{title} (Acc:{acc:.2f}, F1:{f1:.2f})",
    xticklabels=group_labels,
    yticklabels=group_labels,
    font_size = 20
)




# Majority voting
fusion = MajorityVotingFusion(df_part_3)
df_fused = fusion.fuse()
acc, f1, cm = fusion.evaluate(df_fused)
df_fused.to_csv("experiment_12.csv")

# Normalize each column (avoid division by zero)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(cm, row_sums, where=row_sums != 0)

# Get diagonal (accuracy for each class)
per_class_accuracy = np.diag(cm_normalized)

# # Corresponding label
# for i, acc_i in enumerate(per_class_accuracy):
#     print(f"Class {group_labels[i]} accuracy: {acc_i:.2%}")

# Corresponding label
values = []
for i, acc_i in enumerate(per_class_accuracy):
    print(f"Class {group_labels[i]} accuracy: {acc_i:.2%}")
    values.append(f"{acc_i:.4f}")  # or use .2% depending on your needs

# Excel one column (tab-separated)
excel_row = "\t".join(values)
print("\nExcel (row):")
print(excel_row)

title = "Exp 12. V+A+T"
chart_helper.plot_confusion_matrix(
    cm,
    ticklabels=group_labels,
    show_percentage=True,
    task_name="",
    title=f"{title} (Acc:{acc:.2f}, F1:{f1:.2f})",
    xticklabels=group_labels,
    yticklabels=group_labels,
    font_size = 20
)


#============================


# Majority voting
fusion = MajorityVotingFusion(df_part_4)
df_fused = fusion.fuse()
acc, f1, cm = fusion.evaluate(df_fused)
df_fused.to_csv("experiment_13.csv")


# Normalize each column (avoid division by zero)
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(cm, row_sums, where=row_sums != 0)

# Get diagonal (accuracy for each class)
per_class_accuracy = np.diag(cm_normalized)

# Corresponding label
values = []
for i, acc_i in enumerate(per_class_accuracy):
    print(f"Class {group_labels[i]} accuracy: {acc_i:.2%}")
    values.append(f"{acc_i:.4f}")  # or use .2% depending on your needs

# Excel one column (tab-separated)
excel_row = "\t".join(values)
print("\nExcel (row):")
print(excel_row)

title = "Exp 13. V+A+T+P"
chart_helper.plot_confusion_matrix(
    cm,
    ticklabels=group_labels,
    show_percentage=True,
    task_name="",
    title=f"{title} (Acc:{acc:.2f}, F1:{f1:.2f})",
    xticklabels=group_labels,
    yticklabels=group_labels,
    font_size = 20
)
